# SU(3) $O(u^6)$ glueball rest mass: completed verification and recomputation driver

This notebook replaces the incomplete one-cell prototype. It now:

- uses the correct Stage-0 plaquette encoding/decoding convention;
- verifies the final Stage-0 and Stage-1 archives by SHA-256 and exact counts;
- runs the portable internal-release verifier;
- independently checks the folded sixth-order dominant-anchor contribution;
- reports the exact internally certified $m_6$ and normalized $c_6$;
- provides a real resumable `full-contract` mode when the five hashed large intermediate files are supplied.

The immediate result is a **verified internal certificate result**. A full independent re-contraction requires the five exact intermediate artifacts listed by the final readiness report.

In [1]:
#!/usr/bin/env python3
"""Trustworthy SU(3) O(u^6) glueball-rest-mass driver.

This replaces the incomplete notebook stub. It has two computation levels:

1. ``auto`` / ``certified-result``
   Verifies the Stage-0 archive, Stage-1 local release, and the portable internal
   exact release, then reports the exact stored m6 and ratio coefficient. This is
   a certificate verification, not an independent re-contraction.

2. ``full-contract``
   Runs the supplied resumable exact contraction source when all five large
   intermediate artifacts are present. Their SHA-256 hashes are checked before
   any contraction is launched. This mode is a real recomputation of the final
   205,699 nonzero Gamma blocks.

Coupling convention: u = beta_lat/6 = 1/g_H^4.
"""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import itertools
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from fractions import Fraction as F
from pathlib import Path
from typing import Iterable, Sequence

VERSION = "2026-06-15-complete-driver-v2"
SEARCH_ROOTS = [Path("/content"), Path("/mnt/data"), Path.cwd()]
WORK_DEFAULT = Path("/content/SU3_Y6_M6_WORK") if Path("/content").exists() else Path("/mnt/data/SU3_Y6_M6_WORK")

EXPECTED_M6 = F(
    -156998370765216917515896262601525405897211506214753116643443873,
    4880681791275629050759264798095652027950878794719744000000,
)
EXPECTED_C6 = F(
    -4270353824428899200786427191557487127249971701568661364147801,
    265509089445394220361304005016403470320527806432754073600,
)
RATIO_C6_CONSTANT = F(
    1181646977233006828729169209802562361069278851250351799,
    168641444007491247688836385300053017225944999004544000000,
)
KNOWN_M = {
    0: F(8, 3),
    1: F(1),
    2: F(11, 306),
    3: F(-109151, 249696),
    4: F(-20721577909065127111, 7250590288602460800),
    5: F(-866236750503342026253096691057, 1169668083793811403447133488000),
    6: EXPECTED_M6,
}
EXPECTED_COUNTS = {
    "ordered_words": 3_094_806,
    "triality_survivors": 21_175,
    "feasible_orbits": 3_525_818,
    "gamma_topology_blocks": 264_910,
    "nonzero_gamma_blocks": 205_699,
    "global_path_choices": 10_907_384,
    "energy_groups": 2_579_929,
    "nonzero_raw_energy_groups": 1_186_354,
}
FULL_INPUT_HASHES = {
    "Y6_CLASS_ENERGY_SPECTRA.bin": "6407c485dd8322cecdcc444f7c514731a877153564b0398bc64ba95829745d14",
    "Y6_ENERGY_CLASSES.tsv": "d74130cb6e265d2f35c1d3feabac88a1e1e48800d1de439fdbf7e76882d8fa9d",
    "Y6_EXACT_LOCAL_PATH_TENSORS.json.gz": "4d2d5a5ea99ef42b3e27b00404c191d16a840f03eabcb11e94882c0542fbb8a1",
    "Y6_GAMMA_TOPOLOGY_BLOCKS.tsv": "6eff742597123d7fd0f7e632d8f063a2205302a207aafbfa981124e8a24d86a0",
    "Y6_GLOBAL_FOLDED_WEIGHT_CATALOG.tsv": "83dad16b5700a76edd2e5653e620e75de55fc277fcec8f0d0410100b7b85429f",
}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def find_file(name: str, roots: Sequence[Path] = SEARCH_ROOTS) -> Path | None:
    """Find common Colab artifacts without recursively walking huge shard trees."""
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for pattern in (f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"):
            for match in root.glob(pattern):
                if match.is_file():
                    return match
    return None


def require_file(name: str) -> Path:
    p = find_file(name)
    if p is None:
        raise FileNotFoundError(f"Required file not found: {name}")
    return p


def extract_once(archive: Path, dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    marker = dest / f".{archive.name}.extracted"
    digest = sha256(archive)
    if marker.exists() and marker.read_text().strip() == digest:
        return dest
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(dest)
    marker.write_text(digest + "\n")
    return dest


def count_tsv_rows(path: Path) -> int:
    with path.open("rb") as f:
        return max(0, sum(1 for _ in f) - 1)


# ---------------------------------------------------------------------------
# Exact local algebra and folded-weight checks
# ---------------------------------------------------------------------------
def su3_c2_num(ir: tuple[int, int]) -> int:
    p, q = ir
    return p * p + q * q + p * q + 3 * p + 3 * q


def fuse(ir: tuple[int, int], token: int) -> tuple[tuple[int, int], ...]:
    p, q = ir
    if token == 1:
        out = [(p + 1, q)]
        if p:
            out.append((p - 1, q + 1))
        if q:
            out.append((p, q - 1))
    elif token == -1:
        out = [(p, q + 1)]
        if q:
            out.append((p + 1, q - 1))
        if p:
            out.append((p - 1, q))
    else:
        out = [ir]
    return tuple(out)


def fusion_path_counter(tokens: Sequence[int]) -> Counter[tuple[int, ...]]:
    """Return exact singlet histories of C2*3 at each event."""
    states: dict[tuple[int, int], Counter[tuple[int, ...]]] = {(0, 0): Counter({(): 1})}
    for token in tokens:
        nxt: dict[tuple[int, int], Counter[tuple[int, ...]]] = defaultdict(Counter)
        for ir, histories in states.items():
            for ir2 in fuse(ir, token):
                e = su3_c2_num(ir2)
                for hist, mult in histories.items():
                    nxt[ir2][hist + (e,)] += mult
        states = nxt
    return states.get((0, 0), Counter())


def singlet_multiplicity(n_fund: int, n_antifund: int) -> int:
    states = Counter({(0, 0): 1})
    for token in (1,) * n_fund + (-1,) * n_antifund:
        nxt = Counter()
        for ir, mult in states.items():
            for ir2 in fuse(ir, token):
                nxt[ir2] += mult
        states = nxt
    return states[(0, 0)]


def complete_homogeneous(values: Sequence[F], degree: int) -> F:
    dp = [F(0)] * (degree + 1)
    dp[0] = F(1)
    for x in values:
        for k in range(1, degree + 1):
            dp[k] += x * dp[k - 1]
    return dp[degree]


def folded_coefficient(denominators: Sequence[F]) -> F:
    nonzero = [d for d in denominators if d]
    z = len(denominators) - len(nonzero)
    if not nonzero:
        return F(0)
    return F((-1) ** z, z + 1) * complete_homogeneous([1 / d for d in nonzero], z) / math.prod(nonzero)


PLANES = ((0, 1), (0, 2), (1, 2))


def decode_plaquette(q: int) -> tuple[int, int, int, int, int]:
    """Faithful Stage-0 V2 decoder. The old notebook incorrectly used offset 24."""
    x = (q & 63) - 32
    y = ((q >> 6) & 63) - 32
    z = ((q >> 12) & 63) - 32
    plane = (q >> 18) & 3
    if plane >= len(PLANES):
        raise ValueError(f"invalid plane code {plane} in {q}")
    a, b = PLANES[plane]
    return x, y, z, a, b


def encode_plaquette(p: Sequence[int]) -> int:
    x, y, z, a, b = map(int, p)
    try:
        plane = PLANES.index((a, b))
    except ValueError as exc:
        raise ValueError(f"invalid orientation {(a, b)}") from exc
    if not all(-31 <= v <= 31 for v in (x, y, z)):
        raise ValueError("coordinate out of Stage-0 range")
    return (x + 32) | ((y + 32) << 6) | ((z + 32) << 12) | (plane << 18)


def selftest() -> dict:
    expected_mult = {
        (1, 1): 1,
        (0, 3): 1,
        (3, 0): 1,
        (2, 2): 2,
        (1, 4): 3,
        (4, 1): 3,
        (0, 6): 5,
        (6, 0): 5,
        (3, 3): 6,
        (2, 5): 11,
        (5, 2): 11,
        (1, 7): 21,
        (7, 1): 21,
        (4, 4): 23,
    }
    got = {k: singlet_multiplicity(*k) for k in expected_mult}
    assert got == expected_mult, (got, expected_mult)

    samples = [124767, 133152, 649120, 653280]
    for q in samples:
        assert encode_plaquette(decode_plaquette(q)) == q

    e6 = (32, 16, 32, 16, 32)
    ds = tuple(F(16 - e, 6) for e in e6)
    anchor_weight = folded_coefficient(ds)
    assert anchor_weight == F(-243, 16384)
    one = 65208 * anchor_weight
    eight = 8 * one
    assert one == F(-1980693, 2048)
    assert eight == F(-1980693, 256)

    assert EXPECTED_M6 / 2 + RATIO_C6_CONSTANT == EXPECTED_C6
    return {
        "passed": True,
        "version": VERSION,
        "fusion_families_checked": len(expected_mult),
        "geometry_roundtrips": len(samples),
        "dominant_anchor_folded_weight": str(anchor_weight),
        "dominant_anchor_eight_block_total": str(eight),
        "ratio_identity": True,
    }


# ---------------------------------------------------------------------------
# Input and release verification
# ---------------------------------------------------------------------------
def verify_stage0(work: Path) -> dict:
    archive = require_file("SU3_Y6_STAGE0_ESSENTIAL_RESULTS.zip")
    out = extract_once(archive, work / "stage0")
    root = out / "SU3_Y6_RUN"
    ordered = root / "final/y6_ordered_transition_words.tsv"
    survivors = root / "final/y6_triality_survivors.tsv"
    checks = root / "STAGE0_SHA256SUMS.txt"
    assert ordered.is_file() and survivors.is_file() and checks.is_file()

    expected = {}
    for line in checks.read_text().splitlines():
        if not line.strip():
            continue
        digest, rel = line.split("  ", 1)
        expected[Path(rel).name] = digest
    actual = {
        ordered.name: sha256(ordered),
        survivors.name: sha256(survivors),
    }
    assert actual[ordered.name] == expected[ordered.name]
    assert actual[survivors.name] == expected[survivors.name]
    ordered_rows = count_tsv_rows(ordered)
    survivor_rows = count_tsv_rows(survivors)
    assert ordered_rows == EXPECTED_COUNTS["ordered_words"]
    assert survivor_rows == EXPECTED_COUNTS["triality_survivors"]
    return {
        "passed": True,
        "archive": str(archive),
        "archive_sha256": sha256(archive),
        "ordered_words": ordered_rows,
        "triality_survivors": survivor_rows,
        "ordered_words_sha256": actual[ordered.name],
        "triality_survivors_sha256": actual[survivors.name],
        "note": "The stale ordered_merge.log and signature_census.log are ignored.",
    }


def verify_stage1(work: Path) -> dict:
    archive = require_file("SU3_Y6_STAGE1_LOCAL_EXACT_V1_RELEASE.zip")
    out = extract_once(archive, work / "stage1") / "SU3_Y6_STAGE1_LOCAL_EXACT_V1"
    cert = json.loads((out / "Y6_LOCAL_EXACT_PATH_CERTIFICATE.json").read_text())
    feasibility = json.loads((out / "Y6_STAGE1_FEASIBILITY_CERTIFICATE.json").read_text())
    library = out / "Y6_LOCAL_EXACT_PATH_LIBRARY.json.gz"
    catalog = out / "Y6_LOCAL_TOKEN_SIGNATURE_CATALOG.tsv"
    assert sha256(library) == cert["library_sha256"]
    assert sha256(catalog) == cert["catalog_sha256"]
    assert cert["counts"]["feasible_patterns"] == 925
    assert feasibility["feasible_orbits"] == EXPECTED_COUNTS["feasible_orbits"]
    return {
        "passed": True,
        "archive": str(archive),
        "library_sha256": sha256(library),
        "catalog_sha256": sha256(catalog),
        "patterns": cert["counts"]["patterns"],
        "feasible_patterns": cert["counts"]["feasible_patterns"],
        "distinct_local_energy_vectors": cert["complexity"]["distinct_local_energy_vectors"],
        "feasible_orbits": feasibility["feasible_orbits"],
        "rejected_orbits": feasibility["rejected_orbits"],
    }


def verify_internal_release(work: Path) -> dict:
    archive = require_file("SU3_Y6_M6_EXACT_INTERNAL_V1_RELEASE.zip")
    out = extract_once(archive, work / "m6_release") / "SU3_Y6_M6_EXACT_INTERNAL_V1"
    proc = subprocess.run(
        [sys.executable, str(out / "ENGINE_SHELL6_verify_release.py")],
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"release verifier failed\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}")
    exact = json.loads((out / "certificates/Y6_M6_EXACT_CERTIFICATE.json").read_text())
    audit = json.loads((out / "certificates/Y6_M6_INTERNAL_AUDIT_CERTIFICATE.json").read_text())
    assert F(exact["m6"]) == EXPECTED_M6
    assert F(audit["ratio_c6"]) == EXPECTED_C6
    return {
        "passed": True,
        "archive": str(archive),
        "archive_sha256": sha256(archive),
        "verifier_stdout": proc.stdout.strip(),
        "m6": exact["m6"],
        "m6_decimal": exact["m6_decimal"],
        "ratio_c6": audit["ratio_c6"],
        "ratio_c6_decimal": audit["ratio_c6_decimal"],
        "status": audit["status"],
    }


def dominant_anchor_audit(work: Path) -> dict:
    archive = require_file("SU3_Y6_M6_EXACT_INTERNAL_V1_RELEASE.zip")
    out = extract_once(archive, work / "m6_release") / "SU3_Y6_M6_EXACT_INTERNAL_V1"
    cert = json.loads((out / "certificates/Y6_M6_DOMINANT_ANCHOR_CERTIFICATE.json").read_text())
    e6 = tuple(cert["blocks"][0]["E6"])
    ds = tuple(F(16 - e, 6) for e in e6)
    folded = folded_coefficient(ds)
    assert folded == F(cert["blocks"][0]["folded_coefficient"])
    total = F(0)
    for block in cert["blocks"]:
        assert tuple(block["E6"]) == e6
        c = int(block["gamma_coefficient"]) * F(block["raw_color_amplitude"]) * folded
        assert c == F(block["contribution"])
        total += c
    assert total == F(cert["combined_contribution"])
    return {
        "passed": True,
        "blocks": len(cert["blocks"]),
        "E6": list(e6),
        "folded_coefficient": str(folded),
        "combined_contribution": str(total),
        "combined_decimal": float(total),
    }


# ---------------------------------------------------------------------------
# Full final contraction when the large exact intermediate files are supplied
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class FullInputs:
    topology: Path
    classes: Path
    spectra: Path
    weights: Path
    paths: Path


def locate_full_inputs() -> tuple[FullInputs | None, list[str]]:
    found = {name: find_file(name) for name in FULL_INPUT_HASHES}
    missing = [name for name, path in found.items() if path is None]
    if missing:
        return None, missing
    return FullInputs(
        topology=found["Y6_GAMMA_TOPOLOGY_BLOCKS.tsv"],  # type: ignore[arg-type]
        classes=found["Y6_ENERGY_CLASSES.tsv"],  # type: ignore[arg-type]
        spectra=found["Y6_CLASS_ENERGY_SPECTRA.bin"],  # type: ignore[arg-type]
        weights=found["Y6_GLOBAL_FOLDED_WEIGHT_CATALOG.tsv"],  # type: ignore[arg-type]
        paths=found["Y6_EXACT_LOCAL_PATH_TENSORS.json.gz"],  # type: ignore[arg-type]
    ), []


def verify_full_input_hashes(inputs: FullInputs) -> dict:
    mapping = {
        inputs.topology.name: inputs.topology,
        inputs.classes.name: inputs.classes,
        inputs.spectra.name: inputs.spectra,
        inputs.weights.name: inputs.weights,
        inputs.paths.name: inputs.paths,
    }
    actual = {name: sha256(path) for name, path in mapping.items()}
    mismatches = {
        name: {"expected": FULL_INPUT_HASHES[name], "actual": actual[name]}
        for name in FULL_INPUT_HASHES
        if actual[name] != FULL_INPUT_HASHES[name]
    }
    if mismatches:
        raise ValueError(f"Full-input SHA-256 mismatch: {json.dumps(mismatches, indent=2)}")
    return actual


def full_contract(work: Path, threads: int = 2, max_shards: int = 0) -> dict:
    inputs, missing = locate_full_inputs()
    if inputs is None:
        raise FileNotFoundError(
            "Full independent contraction cannot start because these exact intermediate files are missing:\n  - "
            + "\n  - ".join(missing)
        )
    hashes = verify_full_input_hashes(inputs)
    archive = require_file("SU3_Y6_M6_EXACT_INTERNAL_V1_RELEASE.zip")
    release = extract_once(archive, work / "m6_release") / "SU3_Y6_M6_EXACT_INTERNAL_V1"
    source = release / "source/y6_gamma_contraction_batched_resumable.py"
    root = work / "full_contraction"
    cmd = [
        sys.executable,
        str(source),
        "--topology", str(inputs.topology),
        "--classes", str(inputs.classes),
        "--spectra", str(inputs.spectra),
        "--weights", str(inputs.weights),
        "--paths", str(inputs.paths),
        "--root", str(root),
        "--threads", str(threads),
    ]
    if max_shards:
        cmd += ["--max-shards", str(max_shards)]
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    cert_path = root / "Y6_M6_EXACT_CERTIFICATE.json"
    if not cert_path.exists():
        return {
            "passed": True,
            "partial": True,
            "message": "Partial resumable run completed. Re-run the same command to continue.",
            "root": str(root),
            "input_hashes": hashes,
        }
    cert = json.loads(cert_path.read_text())
    assert F(cert["m6"]) == EXPECTED_M6
    return {
        "passed": True,
        "partial": False,
        "root": str(root),
        "m6": cert["m6"],
        "m6_decimal": cert["m6_decimal"],
        "input_hashes": hashes,
        "certificate": str(cert_path),
    }


def result_summary() -> dict:
    series = {f"m{k}": str(v) for k, v in KNOWN_M.items()}
    return {
        "coupling": "u = beta_lat/6 = 1/g_H^4",
        "channel": "SU(3) one-flux C-odd 1^{+-} rest mass at Gamma",
        "series_coefficients": series,
        "m6": str(EXPECTED_M6),
        "m6_decimal": float(EXPECTED_M6),
        "ratio_c6": str(EXPECTED_C6),
        "ratio_c6_decimal": float(EXPECTED_C6),
        "sqrt6_ratio_c6_decimal": math.sqrt(6) * float(EXPECTED_C6),
        "status": "internally exact certificate result; independent full re-contraction requires the five hashed intermediate files",
    }


def print_json(label: str, payload: dict) -> None:
    print(f"\n=== {label} ===")
    print(json.dumps(payload, indent=2, sort_keys=True))


def auto(work: Path) -> dict:
    work.mkdir(parents=True, exist_ok=True)
    report = {
        "selftest": selftest(),
        "stage0": verify_stage0(work),
        "stage1": verify_stage1(work),
        "release": verify_internal_release(work),
        "dominant_anchor": dominant_anchor_audit(work),
        "result": result_summary(),
    }
    inputs, missing = locate_full_inputs()
    report["full_recompute_readiness"] = {
        "ready": inputs is not None,
        "missing": missing,
        "required_hashes": FULL_INPUT_HASHES,
    }
    for key, value in report.items():
        print_json(key, value)
    return report


def main(argv: Sequence[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument(
        "mode",
        nargs="?",
        default="auto",
        choices=["auto", "selftest", "verify-stage0", "verify-stage1", "verify-release", "anchor", "result", "full-contract"],
    )
    ap.add_argument("--work", type=Path, default=WORK_DEFAULT)
    ap.add_argument("--threads", type=int, default=2)
    ap.add_argument("--max-shards", type=int, default=0)
    args = ap.parse_args(argv)
    args.work.mkdir(parents=True, exist_ok=True)

    if args.mode == "auto":
        auto(args.work)
    elif args.mode == "selftest":
        print_json("selftest", selftest())
    elif args.mode == "verify-stage0":
        print_json("stage0", verify_stage0(args.work))
    elif args.mode == "verify-stage1":
        print_json("stage1", verify_stage1(args.work))
    elif args.mode == "verify-release":
        print_json("release", verify_internal_release(args.work))
    elif args.mode == "anchor":
        print_json("dominant_anchor", dominant_anchor_audit(args.work))
    elif args.mode == "result":
        print_json("result", result_summary())
    elif args.mode == "full-contract":
        print_json("full_contract", full_contract(args.work, args.threads, args.max_shards))
    return 0




In [2]:
# Runs every check that can be completed from the uploaded archives.
REPORT = auto(WORK_DEFAULT)


=== selftest ===
{
  "dominant_anchor_eight_block_total": "-1980693/256",
  "dominant_anchor_folded_weight": "-243/16384",
  "fusion_families_checked": 14,
  "geometry_roundtrips": 4,
  "passed": true,
  "ratio_identity": true,
  "version": "2026-06-15-complete-driver-v2"
}

=== stage0 ===
{
  "archive": "/mnt/data/SU3_Y6_STAGE0_ESSENTIAL_RESULTS.zip",
  "archive_sha256": "4360b13079b942738408161fa8f45f80e449d2c2e7b670277449cb45ef4ac5db",
  "note": "The stale ordered_merge.log and signature_census.log are ignored.",
  "ordered_words": 3094806,
  "ordered_words_sha256": "aa951806f830718e4a9c53b2542c26669bcde2770b0e335739a8e5370b389db4",
  "passed": true,
  "triality_survivors": 21175,
  "triality_survivors_sha256": "57e988a02424cd2731685704227b16cd56702c9bd20af2e583f88a894cfb0cf6"
}

=== stage1 ===
{
  "archive": "/mnt/data/SU3_Y6_STAGE1_LOCAL_EXACT_V1_RELEASE.zip",
  "catalog_sha256": "3cf254add70652df8f5468a5881d15acdf4b8995cddf8f58e9c15490efc72f00",
  "distinct_local_energy_vectors"

## Optional full exact re-contraction

The following cell launches the 205,699-block resumable exact contraction only when all five required files are present and match their certified SHA-256 hashes. It never substitutes or reconstructs a missing file silently.

In [3]:
RUN_FULL_CONTRACTION = False  # change to True only after uploading all five exact intermediate files

if RUN_FULL_CONTRACTION:
    FULL_REPORT = full_contract(WORK_DEFAULT, threads=2)
    print_json("full_contract", FULL_REPORT)
else:
    _, missing = locate_full_inputs()
    print("Full contraction not launched.")
    print("Missing files:" if missing else "All full-contraction inputs are present.")
    for name in missing:
        print(" -", name)

Full contraction not launched.
Missing files:
 - Y6_CLASS_ENERGY_SPECTRA.bin
 - Y6_ENERGY_CLASSES.tsv
 - Y6_EXACT_LOCAL_PATH_TENSORS.json.gz
 - Y6_GAMMA_TOPOLOGY_BLOCKS.tsv
 - Y6_GLOBAL_FOLDED_WEIGHT_CATALOG.tsv
